In [1]:
import numpy as np
import optuna
import random
import time
#from
from PIL import Image
from pathlib import Path

#pytorch stuff
import torch
import torch.nn as nn
import torchvision.transforms as transforms
#from
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import resnet18, ResNet18_Weights

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("Running on GPU")
else:
    device = torch.device('cpu')
    print("Running on CPU")

Running on GPU


## Load the dataset

In [3]:
#### DATASET PREPROCESSING #####
img_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
map_transf = transforms.Compose([transforms.Resize((480,640)), transforms.ToTensor()]) #transforms.Normalize((0.5,), (0.5,))])

In [4]:
#### CREATE DATASET ####
class create_dataset(Dataset):
    """ Custom dataset for loading input images and their optional ground-truth saliency maps. """
    def __init__(self, image_paths, map_paths=None, image_transform=img_transf, map_transform=map_transf):
        self.image_paths = image_paths
        self.map_paths = map_paths
        self.image_transform = image_transform
        self.map_transform = map_transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        
        image = Image.open(self.image_paths[idx]).convert("RGB")

        if self.image_transform is not None:
            image = self.image_transform(image)

        # no groundtruth
        if self.map_paths is None:
            return image

        saliency_map = Image.open(self.map_paths[idx]).convert("L")

        if self.map_transform is not None:
            saliency_map = self.map_transform(saliency_map)

        return image, saliency_map

def create_path(path):
    """ This function return a sorted list of path objects contained in the given directory. """
    return sorted(Path(path).iterdir())

In [5]:
#### PATHS ####
train_images_path = create_path("dataset/images/train/")
train_maps_path = create_path("dataset/maps/train/")

val_images_path = create_path("dataset/images/val/")
val_maps_path = create_path("dataset/maps/val/")

In [6]:
#training set 10k
training_set = create_dataset(train_images_path, train_maps_path)

#validation set 5k
full_validation_set = create_dataset(val_images_path, val_maps_path)

#seed for repeating the same experiment with all models, namely the same sets
rng = np.random.default_rng(42)
indices = rng.permutation(len(full_validation_set))
split = int(len(indices) * 0.6)

#3k validation
validation_set = Subset(full_validation_set, indices[:split].tolist())

#2k test
test_set = Subset(full_validation_set, indices[split:].tolist())

## MLResNet Model

In [7]:
#This is basically the same encoder used in SSResNet-18, the difference is that we extract the features at different depths in the forward pass
class MLresnet18_encoder(nn.Module):
    """Encoder based on resnet18.
       Features are extracted at three different levels.
       The encoder can be frozen by setting frozen=True.
       Pretrained ImageNet weights can be used by setting pretrained=True. """

    def __init__(self, pretrained=False, frozen=False):
        super().__init__()

        #weights choice
        if pretrained:
            weights = ResNet18_Weights.IMAGENET1K_V1
        else:
            weights = None

        resnet = resnet18(weights=weights)

        #initial layer
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool

        #residual layers
        self.layer1 = resnet.layer1    # H/4 x W/4
        self.layer2 = resnet.layer2    # H/8 x W/8
        self.layer3 = resnet.layer3    # H/16 x W/16
        self.layer4 = resnet.layer4    # H/16 x W/16
        self.layer4[0].conv1.stride = (1, 1) # set stride 1 instead of 2 (originally was /32, but i set the stride 1)
        self.layer4[0].downsample[0].stride = (1, 1) # skip connection with stride 1 instead of 2


        self.frozen = frozen #frozen state for the encoder
        if self.frozen:
            self.freeze()

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        feature1 = self.layer1(x)
        feature2 = self.layer2(feature1)
        layer3_output = self.layer3(feature2)
        feature3 = self.layer4(layer3_output)
        return feature1, feature2, feature3

    def freeze(self):
        """ Freeze all encoder parameters and set encoder in eval mode. """
        self.frozen = True
        for p in self.parameters():
            p.requires_grad = False
        self.eval()

    def unfreeze(self):
        """ unfreeze encoder parameters."""
        self.frozen = False
        for p in self.parameters():
            p.requires_grad = True
        self.train()

    def train(self, mode=True):
        """ Keep the encoder in eval mode when is frozen to prevent BN updates."""
        super().train(mode)
        if self.frozen:
            super().train(False)
        return self

In [8]:
class decoder(nn.Module):
    def __init__(self):
        super().__init__()

        #1x1 channel projections
        #they adapt resnet channels to the dimensionalities used by the decoders (it's more clear in the forward pass)
        self.projection1 = nn.Conv2d(64, 256, kernel_size=1)
        self.projection2 = nn.Conv2d(128, 512, kernel_size=1)
        self.projection3 = nn.Conv2d(512, 512, kernel_size=1)

        #decoder 1: H/4 -> H
        self.decoder1_up_layer1 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder1_up_layer2 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier1 = nn.Conv2d(64, 1, kernel_size=1)

        #decoder 2: H/8 -> H
        self.decoder2_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer2 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder2_up_layer3 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier2 = nn.Conv2d(64, 1, kernel_size=1)

        #decoder 3: H/16 -> H
        self.decoder3_up_layer1 = nn.Sequential(nn.ConvTranspose2d(512, 512, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer2 = nn.Sequential(nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer3 = nn.Sequential(nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.decoder3_up_layer4 = nn.Sequential(nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1), nn.ReLU(inplace=True))
        self.classifier3 = nn.Conv2d(64, 1, kernel_size=1)

        #learned fusion of the three saliency logits
        self.fusion = nn.Conv2d(3, 1, kernel_size=1)


    def forward(self, features):
        feature1, feature2, feature3 = features

        #channel projection
        feature1 = self.projection1(feature1)    # 64 -> 256
        feature2 = self.projection2(feature2)    # 128 -> 512
        feature3 = self.projection3(feature3)    # 512 -> 512

        #decoder 1
        x1 = self.decoder1_up_layer1(feature1)
        x1 = self.decoder1_up_layer2(x1)
        logit1 = self.classifier1(x1)

        #decoder 2
        x2 = self.decoder2_up_layer1(feature2)
        x2 = self.decoder2_up_layer2(x2)
        x2 = self.decoder2_up_layer3(x2)
        logit2 = self.classifier2(x2)

        #decoder 3
        x3 = self.decoder3_up_layer1(feature3)
        x3 = self.decoder3_up_layer2(x3)
        x3 = self.decoder3_up_layer3(x3)
        x3 = self.decoder3_up_layer4(x3)
        logit3 = self.classifier3(x3)

        #fusion
        multi_level_logits = torch.cat((logit1, logit2, logit3), dim=1)
        fused_logit = self.fusion(multi_level_logits)

        return {"level1": logit1, "level2": logit2, "level3": logit3, "fused": fused_logit}


In [ ]:
### DS loss implemented as in the reference paper
class deep_supervision_loss(nn.Module):
    """Deep-supervision BCE loss.
       Training objective:
           total_loss = (BCE_level1 + BCE_level2 + BCE_level3) / 3 + BCE_fusion
       The model returns logits. """

    def __init__(self):
        super().__init__()
        self.BCE = nn.BCEWithLogitsLoss()

    def forward(self, outputs, maps):
        loss1 = self.BCE(outputs["level1"], maps)
        loss2 = self.BCE(outputs["level2"], maps)
        loss3 = self.BCE(outputs["level3"], maps)
        fusion_loss = self.BCE(outputs["fused"], maps)
        intermediate_loss = (loss1 + loss2 + loss3) / 3
        total_loss = (intermediate_loss + fusion_loss)
        return total_loss


In [9]:
class encoder_decoder(nn.Module):
    def __init__(self, pretrained=False, frozen=False):
        super().__init__()
        self.encoder = MLresnet18_encoder(pretrained=pretrained, frozen=frozen)
        self.decoder = decoder()

    def forward(self, x):
        # skip gradient tracking when encoder is frozen
        if self.encoder.frozen:
            with torch.no_grad():
                features = self.encoder(x)
        else:
            features = self.encoder(x)

        output = self.decoder(features)
        return output

    def freeze_encoder(self):
        self.encoder.freeze()

    def unfreeze_encoder(self):
        self.encoder.unfreeze()


In [11]:
#### SANITY CHECK ####

#img, map = training_set[0]
#
#print("train samples:", len(training_set))
#print("validation samples:", len(validation_set))
#print("test samples:", len(test_set))
#print("image shape:", img.shape)
#print("map shape:", map.shape)
#
#test_model = encoder_decoder(pretrained=False, frozen=False).to(device)
#
#with torch.no_grad():
#    outputs = test_model(img.unsqueeze(0).to(device))
#
#for name, output in outputs.items():
#    print(name, output.shape)
#
#del test_model


train samples: 10000
validation samples: 3000
test samples: 2000
image shape: torch.Size([3, 480, 640])
map shape: torch.Size([1, 480, 640])
level1 torch.Size([1, 1, 480, 640])
level2 torch.Size([1, 1, 480, 640])
level3 torch.Size([1, 1, 480, 640])
fused torch.Size([1, 1, 480, 640])
Parameters: 13,353,543 total
Trainable: 13,353,543


## Optuna objective and training

In [ ]:
def require_finite(tensor, label):
    """Stop a numerically invalid trial"""
    if not torch.isfinite(tensor).all().item():
        raise FloatingPointError(f"{label}: NaN or Inf detected")


class trainer:
    def __init__(self, model, optimizer, loss_function, device):

        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_function = loss_function
        self.device = device
        self.use_amp = (self.device.type == "cuda")
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.use_amp)

    def train_epoch(self, train_loader):
        self.model.train()
        total_loss = 0
        total_samples = 0

        for images, maps in train_loader:
            images = images.to(self.device, non_blocking=True)
            maps = maps.to(self.device,non_blocking=True)
            self.optimizer.zero_grad(set_to_none=True)
            
            with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):
                outputs = self.model(images)
                # DS loss
                loss = self.loss_function(outputs, maps)
                
            require_finite(loss, "training loss")
         
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()

            batch_size = images.size(0)
            total_loss += (loss.detach().float().item() * batch_size)
            total_samples += batch_size

        return (total_loss / total_samples)


    def validate(self, val_loader):
        self.model.eval()
        total_loss = 0
        total_level1_BCE = 0
        total_level2_BCE = 0
        total_level3_BCE = 0
        total_fused_BCE = 0
        total_samples = 0

        BCE = nn.BCEWithLogitsLoss()

        with torch.no_grad():
            for images, maps in val_loader:
                images = images.to(self.device, non_blocking=True)
                maps = maps.to(self.device, non_blocking=True)

                with torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=self.use_amp):
                    outputs = self.model(images)
                    loss = self.loss_function(outputs, maps)

                    # BCE for each output
                    level1_BCE = BCE(outputs["level1"], maps)
                    level2_BCE = BCE(outputs["level2"], maps)
                    level3_BCE = BCE(outputs["level3"], maps)
                    #final BCE used for comparison with SS
                    fused_BCE = BCE(outputs["fused"], maps)
                require_finite(loss, "validation loss")
                batch_size = images.size(0)
                total_loss += (loss.detach().float().item() * batch_size)
                total_level1_BCE += (level1_BCE.detach().float().item() * batch_size)
                total_level2_BCE += (level2_BCE.detach().float().item() * batch_size)
                total_level3_BCE += (level3_BCE.detach().float().item() * batch_size)
                total_fused_BCE += (fused_BCE.detach().float().item() * batch_size)
                total_samples += batch_size

        metrics = {"val_loss": total_loss / total_samples, "level1_BCE": total_level1_BCE / total_samples, "level2_BCE": total_level2_BCE / total_samples, 
                   "level3_BCE": total_level3_BCE / total_samples, "fused_BCE": total_fused_BCE / total_samples}
        return metrics


    def fit(self, train_loader, val_loader, epochs, trial=None):
        history = {"train_loss": [], "val_loss": [], "level1_BCE": [], "level2_BCE": [], "level3_BCE": [], "fused_BCE": []}
        #best model is selected by final fused BCE
        best_fused_BCE = float("inf")
        best_model_state = None
        best_epoch = None
        best_metrics = None

        for epoch in range(epochs):
            epoch_start = time.perf_counter()

            train_loss = self.train_epoch(train_loader)
            val_metrics = self.validate(val_loader)

                
            val_loss = val_metrics["val_loss"]
            fused_BCE = val_metrics["fused_BCE"]
            history["train_loss"].append(train_loss)


            for key in ["val_loss","level1_BCE", "level2_BCE","level3_BCE", "fused_BCE"]:
                history[key].append(val_metrics[key])

            #best model
            if fused_BCE < best_fused_BCE:

                best_fused_BCE = fused_BCE
                best_epoch = epoch + 1

                best_metrics = (val_metrics.copy())

                best_model_state = {key: value.detach().cpu().clone() for key, value in self.model.state_dict().items()}

            epoch_time = (time.perf_counter() - epoch_start)


            print(f"epoch {epoch + 1}/{epochs} | train total loss: {train_loss:.6f} | val total loss: {val_loss:.6f} | fused BCE: {fused_BCE:.6f} | " 
                  f"best fused BCE: {best_fused_BCE:.6f} | (epoch {best_epoch}) | time: {epoch_time:.1f}s", flush=True)

            print(f"side BCE | level1: {val_metrics['level1_BCE']:.6f} | level2: {val_metrics['level2_BCE']:.6f} | level3: {val_metrics['level3_BCE']:.6f}", flush=True)

            if trial is not None:

                # optuna optimize final bce
                trial.report(fused_BCE, step=epoch)

                if trial.should_prune():
                    print(f"trial {trial.number} PRUNED | epoch={epoch + 1} | fused_BCE={fused_BCE:.6f}", flush=True)
                    raise optuna.TrialPruned()

        if best_metrics is None or best_model_state is None:
            raise FloatingPointError("No valid checkpoint was obtained")

        self.model.load_state_dict(best_model_state)

        return (best_fused_BCE, best_metrics, history)


In [1]:
def objective(trial, pretrained=False, frozen=False):
    """ Optuna objective. It runs an optuna trial and returns the best validation loss. """
    #the best val loss here is the fused BCE
    #same seed for everything in order to maximize reproducibility
    seed = 42

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    #hyperparameters
    batch_size = trial.suggest_categorical("batch_size", [4, 8, 16, 32])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "RMSprop", "NAdam", "SGD"])
    weight_decay = trial.suggest_float( "weight_decay", 1e-8, 1e-3, log=True)

    # loss
    loss_function = deep_supervision_loss()

    #decoder lr
    if optimizer_name == "SGD":
        lr_decoder = trial.suggest_float("lr_decoder", 1e-4, 1e-1, log=True)
    else:
        lr_decoder = trial.suggest_float("lr_decoder", 1e-5, 1e-2, log=True)

    #encoder lr if not frozen
    if not frozen:
        if pretrained:
            if optimizer_name == "SGD":
                lr_encoder = trial.suggest_float("lr_encoder", 1e-5, 1e-2, log=True)
            else:
                lr_encoder = trial.suggest_float("lr_encoder", 1e-6, 1e-4, log=True)
        else:
            #coarser tuning
            if optimizer_name == "SGD":
                lr_encoder = trial.suggest_float("lr_encoder", 1e-4, 1e-1, log=True)
            else:
                lr_encoder = trial.suggest_float("lr_encoder", 1e-5, 1e-2, log=True)

    #data loader
    loader_options = dict(num_workers=12, pin_memory=(device.type == "cuda"), persistent_workers=True, prefetch_factor=4)
    train_loader = DataLoader(training_set, batch_size=batch_size, shuffle=True, **loader_options)
    val_loader = DataLoader(validation_set, batch_size=batch_size, shuffle=False, **loader_options)

    #model
    model = encoder_decoder(pretrained=pretrained, frozen=frozen).to(device)

    #parameter groups
    if frozen:
        # frozen encoder, lr_enc excluded from optimizer
        parameter_groups = [{"params": model.decoder.parameters(), "lr": lr_decoder}]
    else:
        parameter_groups = [{"params": model.encoder.parameters(), "lr": lr_encoder}, {"params": model.decoder.parameters(), "lr": lr_decoder}]

    optimizer_options = {"weight_decay": weight_decay}

    if optimizer_name in ["SGD", "RMSprop"]:
        optimizer_options["momentum"] = trial.suggest_float("momentum", 0.0, 0.95)
    else:
        beta1 = trial.suggest_float("beta1", 0.8, 0.99)
        beta2 = trial.suggest_float("beta2", 0.9, 0.9999)
        optimizer_options["betas"] = (beta1, beta2)

    #history log
    print("\n" + "=" * 84, flush=True)
    print(f"trial {trial.number}", flush=True)
    print(f"pretrained={pretrained} | frozen={frozen} | optimizer={optimizer_name} | batch={batch_size}", flush=True)

    if frozen:
        print(f"lr_decoder={lr_decoder:.2e} | weight_decay={weight_decay:.2e}", flush=True)
    else:
        print(f"lr_encoder={lr_encoder:.2e} | lr_decoder={lr_decoder:.2e} | weight_decay={weight_decay:.2e}", flush=True)

    print(f"optimizer_options={optimizer_options}", flush=True)
    print("=" * 84, flush=True)

    optimizer_class = getattr(torch.optim, optimizer_name)
    optimizer = optimizer_class(parameter_groups, **optimizer_options)
    
    model_trainer = trainer(model=model, optimizer=optimizer, loss_function=loss_function, device=device)

    best_fused_BCE, best_metrics, history = model_trainer.fit(train_loader=train_loader, val_loader=val_loader, epochs=30, trial=trial)

    print(f"\ntrial {trial.number} COMPLETE | best fused BCE: {best_fused_BCE:.6f} | total loss at best epoch: {best_metrics['val_loss']:.6f} ", flush=True)

    return best_fused_BCE

In [ ]:
#to optune de model, set pretrained and frozen configuration and change the study name and the storage, then run .py and hope everything works fine. 
#hopefully, if a FPerror raises, Optuna doesn't stop and it reports it as failed
study = optuna.create_study(study_name="", storage="sqlite:///optuna_studies/",
                            direction="minimize", load_if_exists=True, pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))
study.optimize(lambda trial: objective( trial, pretrained=False, frozen=False), n_trials=25, catch=(FloatingPointError,), gc_after_trial=True)